# SpaceX Falcon 9 - Launch Site Locations with Folium

In [ ]:
# folium already installed in the venv
import folium

In [ ]:
import folium
import pandas as pd
import numpy as np
from folium.plugins import MarkerCluster, MousePosition
from folium.features import DivIcon
from math import sin, cos, sqrt, atan2, radians

In [ ]:
spacex_df = pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv')
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]
launch_sites_df

## Task 1: Mark all launch sites on a map

In [ ]:
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)

for _, row in launch_sites_df.iterrows():
    coord = [row['Lat'], row['Long']]
    folium.Circle(coord, radius=1000, color='#d35400', fill=True).add_child(folium.Popup(row['Launch Site'])).add_to(site_map)
    folium.map.Marker(coord, icon=DivIcon(icon_size=(20, 20), icon_anchor=(0, 0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % row['Launch Site'])).add_to(site_map)
site_map

## Task 2: Mark success/failed launches for each site

In [ ]:
spacex_df['marker_color'] = spacex_df['class'].apply(lambda c: 'green' if c == 1 else 'red')
marker_cluster = MarkerCluster()
site_map.add_child(marker_cluster)
for _, row in spacex_df.iterrows():
    marker = folium.Marker([row['Lat'], row['Long']],
                           icon=folium.Icon(color='white', icon_color=row['marker_color']))
    marker_cluster.add_child(marker)
site_map

## Task 3: Distances between a launch site and its proximities

In [ ]:
site_map.add_child(MousePosition(position='topright', separator=' Long: ', empty_string='NaN',
                                 lng_first=False, num_digits=20, prefix='Lat:',
                                 lat_formatter="function(num) {return L.Util.formatNum(num, 5);};",
                                 lng_formatter="function(num) {return L.Util.formatNum(num, 5);};"))

def calculate_distance(lat1, lon1, lat2, lon2):
    R = 6373.0
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlon, dlat = lon2 - lon1, lat2 - lat1
    a = sin(dlat / 2) ** 2 + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    return R * 2 * atan2(sqrt(a), sqrt(1 - a))

In [ ]:
# Launch site: CCAFS SLC-40. Proximity coordinates are approximate points picked
# on the map with MousePosition; adjust them if you pick more accurate ones.
launch_site_lat, launch_site_lon = 28.56321, -80.57683
proximities = {
    'Coastline': (28.56367, -80.57163),
    'Highway':   (28.56342, -80.57674),
    'Railway':   (28.57205, -80.58527),
    'City (Melbourne)': (28.10473, -80.64531),
}
for name, (lat, lon) in proximities.items():
    d = calculate_distance(launch_site_lat, launch_site_lon, lat, lon)
    folium.Marker([lat, lon], icon=DivIcon(icon_size=(20, 20), icon_anchor=(0, 0),
        html='<div style="font-size: 12; color:#252526;"><b>{:10.2f} KM</b></div>'.format(d))).add_to(site_map)
    folium.PolyLine(locations=[[launch_site_lat, launch_site_lon], [lat, lon]], weight=1).add_to(site_map)
    print(f'{name}: {d:.2f} km')
site_map